In [1]:
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

from omegaconf import OmegaConf
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

import xgboost as xgb


from config import py_config


['RandomForestClassifier', 'LogisticRegression', 'XGBClassifier', 'HistGradientBoostingClassifier', 'ExtraTreesClassifier'] <class 'list'>
['RandomForestClassifier', 'LogisticRegression', 'XGBClassifier', 'HistGradientBoostingClassifier', 'ExtraTreesClassifier'] <class 'omegaconf.listconfig.ListConfig'>
5 <class 'int'>


In [2]:
""" 
Pipeline:
1. Load data
2. Preprocess data
    2.1. Split data into features and target
    2.2. Save "PassengerId" for submission(test set)
    2.3. BaseLine features: move to trash some noise features
    2.4. Split train data to train and validation sets with stratification / train_test_split 
    2.5. Impute missing values with SimpleImputer
    2.6. Scale numerical features with StandardScaler
    2.7. Encode cat features with one-hot encoding
3. Fit models and evaluate with cross-validation
4. Chose the best model and make pradictions on test set
5. Submit results on Kaggle
"""

' \nPipeline:\n1. Load data\n2. Preprocess data\n    2.1. Split data into features and target\n    2.2. Save "PassengerId" for submission(test set)\n    2.3. BaseLine features: move to trash some noise features\n    2.4. Split train data to train and validation sets with stratification / train_test_split \n    2.5. Impute missing values with SimpleImputer\n    2.6. Scale numerical features with StandardScaler\n    2.7. Encode cat features with one-hot encoding\n3. Fit models and evaluate with cross-validation\n4. Chose the best model and make pradictions on test set\n5. Submit results on Kaggle\n'

In [32]:
train = pd.read_csv(py_config.paths.train)
test = pd.read_csv(py_config.paths.test)
# train.head(20)
# test.head(20)
# print(train.columns.to_list())
# print("---------------")
# print(test.columns.to_list())


In [4]:
y = train[py_config.features.target]
X = train.drop(columns=py_config.features.drop + [py_config.features.target])
test_X = test.drop(columns=py_config.features.drop)
# test_passenger_id = test[py_config.features.target]

Тут проверки на соответствие нужным классам и группам

In [5]:
print(type(py_config.features.target), py_config.features.target)
print(type(y), getattr(y, "shape", None))

<class 'str'> Survived
<class 'pandas.core.series.Series'> (891,)


In [6]:
train_only = sorted(set(X.columns) - set(test_X.columns))
test_only = sorted(set(test_X.columns) - set(X.columns))
print("train_only:", train_only)
print("test_only:", test_only)

train_only: []
test_only: []


In [7]:
test_X.head(20)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,34.5,0,0,7.8292,Q
1,3,female,47.0,1,0,7.0000,S
2,2,male,62.0,0,0,9.6875,Q
3,3,male,27.0,0,0,8.6625,S
4,3,female,22.0,1,1,12.2875,S
5,3,male,14.0,0,0,9.2250,S
6,3,female,30.0,0,0,7.6292,Q
7,2,male,26.0,1,1,29.0000,S
8,3,female,18.0,0,0,7.2292,C
9,3,male,21.0,2,0,24.1500,S


Разделили категоральные фичи и числовые

In [8]:
cat_cols = ['Sex', 'Embarked']
num_cols = [c for c in X.columns if c not in cat_cols and c != 'Survived']
num_cols

['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

Тут сделаем заполнение пустых клетов в категориальных фичах и числовых фичах

In [9]:
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
# print(cat_pipeline)
# X_train = cat_pipeline.fit_transform(X[cat_cols])
# print(X_train)
preprocess = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])
# print(type(preprocess))
# print(preprocess)

In [10]:
MODEL_REGISTRY = {
    'LogisticRegression': LogisticRegression,
    'RandomForestClassifier': RandomForestClassifier,
    'ExtraTreesClassifier': ExtraTreesClassifier,
    'HistGradientBoostingClassifier': HistGradientBoostingClassifier,
    'XGBClassifier': xgb.XGBClassifier
}

Build models

In [11]:
print(type(py_config.model.types))
print(type(py_config.model.params))
print(type(py_config.cross_validation))

<class 'omegaconf.listconfig.ListConfig'>
<class 'omegaconf.dictconfig.DictConfig'>
<class 'omegaconf.dictconfig.DictConfig'>


In [12]:
def build_models(py_config):
    models = {}
    names = py_config.model.types
    if names is None:
        raise ValueError('ModelType is not found in config')

    for name in names:
        cls = MODEL_REGISTRY[name]
        param_cfg = py_config.model.params.get(name, {})
        param = OmegaConf.to_container(param_cfg, resolve=True) # Перевели из Dict.conf -> dict
        models[name] = cls(**param)
    return models

In [13]:
models = build_models(py_config)
# print(models.keys())
# print(models)

Cross Validation

In [14]:
cv = StratifiedKFold(n_splits=py_config.cross_validation.n_splits,
                     shuffle=py_config.cross_validation.shuffle,
                     random_state=py_config.cross_validation.random_state)
print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [15]:
'''
Нужно разобраться с Pipeline и и как его правильно применить к кросс-валидации.
'''

'\nНужно разобраться с Pipeline и и как его правильно применить к кросс-валидации.\n'

In [16]:
scores = {}
for name, model in models.items():
    pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])
    cv_score = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
    scores[name] = (cv_score.mean(), cv_score.std())
    print(f'{name}: {cv_score.mean():.4f} ± {cv_score.std():.4f}')


RandomForestClassifier: 0.8339 ± 0.0198
LogisticRegression: 0.7969 ± 0.0184
XGBClassifier: 0.8350 ± 0.0256
HistGradientBoostingClassifier: 0.8350 ± 0.0137
ExtraTreesClassifier: 0.8070 ± 0.0135


In [17]:
best_model_name = max(scores, key=lambda k: scores[k][0])
print(f'Best model: {best_model_name} with accuracy {scores[best_model_name][0]:.4f}')

Best model: XGBClassifier with accuracy 0.8350


In [18]:
best_model = models[best_model_name]
best_pipe = Pipeline(steps=[('prep', preprocess), ('model', best_model)])
best_pipe.fit(X,y)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked'])])),
                ('model',
                 XGBClassifier(base_score...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [19]:
test_pred = best_pipe.predict(test_X)

Небольшая проверка на утечку

In [20]:
print("Survived in train_X?", "Survived" in X.columns)
print("Survived in test_X?", "Survived" in test_X.columns)

print("Survived in num_cols?", "Survived" in num_cols)
print("Survived in cat_cols?", "Survived" in cat_cols)

Survived in train_X? False
Survived in test_X? False
Survived in num_cols? False
Survived in cat_cols? False


In [21]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_pred.astype(int)
})
submission.to_csv('submission.csv', index=False)